In [19]:
from sqlalchemy import create_engine
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import text

In [20]:
load_dotenv()

MYSQL_USER = os.getenv('MYSQL_USER')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD')
MYSQL_DB = os.getenv('MYSQL_DB')
MYSQL_HOST = os.getenv('MYSQL_HOST')

# Define database URI
DATABASE_URI = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{MYSQL_DB}'
engine = create_engine(DATABASE_URI)

In [21]:
def select_for_run_nith(run_id):
    query = text("""SELECT file_id, response, model
                FROM inference_nith
                 WHERE run = :run_id
            """)
        
    return pd.read_sql(query, engine, params={'run_id': run_id})   

In [22]:
runs_definition = {
    79: 5,
    89: 10,
    22: 20
}

In [23]:
# run 1 -> with ditributed custom example
# run 2 -> single file
# run 4 -> for 3 files at the same time CWE 79 -> mistral
# run 5 -> for 3 files for llama 3
# safe runs are always +1
CWE = 22
predictions = select_for_run_nith(runs_definition[CWE])
# predictions = select_for_run_nith(11)

for model, group in predictions.groupby('model'):
    cwe_id = f"CWE-{CWE}"
    filename = f"./data/data_nith_{cwe_id}_{model}.csv"
    
    # Rename 'response' column to 'model_output'
    group = group.rename(columns={'response': 'model_output'})
    group.to_csv(filename, index=False, columns=['file_id', 'model_output'])